In [ ]:
from google.colab import userdata
from pathlib import Path
import os, subprocess

repo = Path('/content/Dissertation')
url = 'https://github.com/520/Dissertation.git'
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('请在 Colab Secrets 中添加 GITHUB_TOKEN')

askpass = Path('/tmp/github_askpass.sh')
askpass.write_text("#!/bin/sh\ncase \"$1\" in\n*Username*) echo x-access-token;;\n*Password*) echo \"$GITHUB_TOKEN\";;\nesac\n")
askpass.chmod(0o700)
env = {**os.environ, 'GITHUB_TOKEN': token, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0'}

def git(*args):
    result = subprocess.run(['git', *map(str, args)], env=env, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if (repo / '.git').is_dir():
    print('仓库已存在，正在与 GitHub main 同步……')
    git('-C', repo, 'remote', 'set-url', 'origin', url)
    git('-C', repo, 'fetch', '--prune', 'origin', 'main')
    git('-C', repo, 'reset', '--hard', 'origin/main')
elif repo.exists():
    raise RuntimeError(f'{repo} 已存在但不是 Git 仓库，请删除或改名后重试')
else:
    print('仓库不存在，正在克隆……')
    git('clone', '--branch', 'main', '--single-branch', url, repo)

os.chdir(repo)
print('同步完成，当前 commit：', git('-C', repo, 'rev-parse', '--short', 'HEAD'))

In [ ]:
import importlib.metadata, importlib.util, subprocess, sys, torch

torch_minor = int(torch.__version__.split('+')[0].split('.')[1])
if torch_minor <= 7:
    torchao_version = '0.15.0'
elif torch_minor == 8:
    torchao_version = '0.16.0'
elif torch_minor <= 11:
    torchao_version = '0.17.0'
else:
    torchao_version = '0.18.0'

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'nvidia-modelopt[torch,onnx]'])
try:
    installed_torchao = importlib.metadata.version('torchao')
    has_pt2e = importlib.util.find_spec('torchao.quantization.pt2e') is not None
except (ImportError, ModuleNotFoundError, importlib.metadata.PackageNotFoundError):
    installed_torchao, has_pt2e = None, False

if installed_torchao != torchao_version or not has_pt2e:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir', '--force-reinstall', '--no-deps', f'torchao=={torchao_version}'])
    print('TorchAO 已安装为', torchao_version, '。请重启 Colab 会话，再重新运行此单元格。')
else:
    print('环境可用：PyTorch', torch.__version__, '| TorchAO', installed_torchao, '| PT2E', has_pt2e)

In [ ]:
%cd /content/Dissertation
!python -m QAT.torchao_qat